# Baliser le corpus avec spaCy

## 1. Importer les bibliothèques

Le script utilise :

- `spaCy` pour détecter les entités nommées ;
- `lxml` pour lire et modifier les fichiers XML ;
- `pathlib` pour gérer les chemins ;
- `joblib` pour traiter plusieurs fichiers en parallèle ;
- `re` pour estimer la langue du texte à analyser.

In [ ]:
import spacy
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from lxml import etree
from joblib import Parallel, delayed

## 2. Définir les chemins et les balises TEI

Les modèles spaCy sont chargés dans chaque worker au moment du traitement d’un fichier. Le chargement n’a donc pas besoin d’être effectué une première fois dans le notebook.

Les étiquettes produites par spaCy sont ensuite converties vers les balises utilisées dans le corpus TEI.

In [ ]:
# Dossier contenant les fichiers XML à annoter.
DIR_V0 = Path("output/v0/")

# Les fichiers annotés sont écrits dans un dossier distinct pour préserver les originaux.
OUTPUT_DIR = Path("output/Vspacy/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Conversion des catégories spaCy vers les balises attendues par le corpus TEI.
LABEL_MAP = {
    "PER": "persName",
    "LOC": "placeName",
    "GPE": "placeName",
}

## 3. Détecter les entités et reconstruire le XML

Le traitement suit quatre étapes :

1. estimer si le segment est plutôt français ou italien ;
2. analyser ce segment avec le modèle spaCy correspondant ;
3. conserver les entités `PER`, `LOC` et `GPE` ;
4. remplacer leur texte par des éléments XML `persName` ou `placeName`.

Dans un élément XML, le texte peut se trouver dans `element.text` avant le premier enfant, ou dans `child.tail` après un enfant. Les deux emplacements doivent donc être traités séparément.

Le parcours des enfants est figé avant les insertions : les nouvelles balises ajoutées ne doivent pas être analysées une seconde fois.

In [ ]:
def detect_lang(text):
    """Choisit le modèle à partir de quelques marqueurs français ou italiens."""
    fr_markers = len(re.findall(
        r"\b(le|la|les|de|du|des|un|une|et|en|je|il|elle|nous|vous|ils)\b",
        text,
        re.IGNORECASE,
    ))
    it_markers = len(re.findall(
        r"\b(il|la|le|di|del|della|un|una|e|in|io|lui|lei|noi|voi|loro)\b",
        text,
        re.IGNORECASE,
    ))
    return "it" if it_markers > fr_markers else "fr"


def annotate_text(text, nlp_fra, nlp_ita):
    """Découpe un segment en texte ordinaire et entités à baliser."""
    # Les segments vides ne nécessitent pas d'appel à spaCy.
    if not text or not text.strip():
        return [{"text": text, "tag": None}]

    # Le modèle est choisi segment par segment, car un fichier peut être multilingue.
    nlp = nlp_ita if detect_lang(text) == "it" else nlp_fra
    doc = nlp(text)

    segments = []
    last = 0
    for ent in doc.ents:
        tag = LABEL_MAP.get(ent.label_)
        # Les entités non prévues par LABEL_MAP ne sont pas annotés.
        if not tag:
            continue
        if ent.start_char > last:
            segments.append({"text": text[last:ent.start_char], "tag": None})
        segments.append({"text": ent.text, "tag": tag})
        last = ent.end_char

    # Conserver le texte situé après la dernière entité détectée.
    segments.append({"text": text[last:], "tag": None})
    return segments


def rebuild_text(element, attr, segments, insert_start=0):
    # Remplace une portion de texte par des enfants XML balisés.
    # Sans entité reconnue, le XML n'a pas besoin d'être reconstruit.
    if not any(segment["tag"] for segment in segments):
        return 0

    if attr == "text":
        # Le texte avant le premier enfant sera recréé progressivement ci-dessous.
        element.text = None
    else:
        # Pour un tail, le texte appartient à l'enfant précédent.
        if insert_start > 0:
            try:
                element[insert_start - 1].tail = None
            except (IndexError, TypeError):
                pass

    inserted = 0
    for seg in segments:
        if seg["tag"]:
            new_el = etree.Element(seg["tag"])
            new_el.text = seg["text"]
            new_el.tail = ""
            element.insert(insert_start + inserted, new_el)
            inserted += 1
        else:
            # Le texte ordinaire est placé avant le premier enfant ou dans le
            # tail de l'enfant précédent, selon sa position dans l'élément.
            pos = insert_start + inserted
            if pos == 0:
                element.text = (element.text or "") + seg["text"]
            else:
                try:
                    element[pos - 1].tail = (element[pos - 1].tail or "") + seg["text"]
                except (IndexError, TypeError):
                    element.text = (element.text or "") + seg["text"]
    return inserted


def process_element(root, nlp_fra, nlp_ita):
    # Faire un parcours préalable évite que les balises ajoutées soient traitées
    # comme si elles appartenaient au XML source.
    all_elements = []
    stack = [root]
    while stack:
        element = stack.pop()
        all_elements.append(element)
        for child in reversed(list(element)):
            stack.append(child)

    for element in all_elements:
        try:
            # rebuild_text insère de nouveaux enfants.
            original_children = list(element)

            # 1. Traiter le texte situé avant le premier enfant.
            if element.text and element.text.strip():
                segments = annotate_text(element.text, nlp_fra, nlp_ita)
                rebuild_text(element, "text", segments, insert_start=0)

            # 2. Traiter uniquement les tails des enfants présents dans le XML source.
            for child in original_children:
                if child.tail and child.tail.strip():
                    # Les insertions précédentes peuvent avoir déplacé la position.
                    idx = list(element).index(child)
                    segments = annotate_text(child.tail, nlp_fra, nlp_ita)
                    child.tail = None
                    rebuild_text(element, "tail", segments, insert_start=idx + 1)

        except Exception as error:
            # Un fichier mal formé ne doit pas interrompre le traitement des autres.
            import traceback
            print(f"  → Erreur sur élément <{element.tag}> : {error}")
            traceback.print_exc()
            continue


def process_file(xml_path, output_dir):
    """Traite un fichier XML dans son worker et écrit sa copie annotée."""
    # Chaque worker charge ses propres modèles.
    nlp_fra = spacy.load("fr_core_news_sm")
    nlp_ita = spacy.load("it_core_news_sm")

    try:
        tree = etree.parse(str(xml_path), etree.XMLParser(remove_blank_text=False))
        process_element(tree.getroot(), nlp_fra, nlp_ita)
        output_path = output_dir / xml_path.name
        tree.write(str(output_path), encoding="utf-8", xml_declaration=True, pretty_print=True)
        return f"✓ {xml_path.name}"
    except Exception as error:
        return f"✗ {xml_path.name} — erreur : {error}"

## 4. Traiter les fichiers en parallèle

Les fichiers XML du dossier d’entrée sont distribués à quatre workers. Chaque worker charge les modèles une seule fois, traite son fichier, puis écrit une copie dans `output/Vspacy/`.

Le nombre de workers peut être diminué si la mémoire disponible est limitée.

In [ ]:
# Chargement des fichiers XML à traiter
xml_files = list(DIR_V0.glob("*.xml"))
print(f"{len(xml_files)} fichiers à traiter avec 4 workers...\n")

# Chaque fichier est traité indépendamment, puis le résultat est écrit dans OUTPUT_DIR.
results = Parallel(n_jobs=4, backend="loky", verbose=0)(
    delayed(process_file)(f, OUTPUT_DIR) for f in xml_files
)

for r in results:
    print(r)
print("\nTerminé.")

19 fichiers à traiter avec 4 workers...

✓ Agucchi_TrattatoPittura.xml
✓ Daret_VieRaphael.xml
✓ DupuyDuGrez_TraitePeinture.xml
✓ Freart_IdeaDellaPerfezione.xml
✓ Lomazzo_Idea.xml
✓ Lomazzo_TraicteProportion.xml
✓ Marino_DicerieSacre.xml
✓ Monier_HistoireArtsRapportDessein.xml
✓ Pader_LaPeintureParlante.xml
✓ Pader_SongeEnigmatique.xml
✓ Piles_AbregeViePeintres.xml
✓ Piles_ConversationsConnaissancePeinture.xml
✓ Piles_CoursPeinture.xml
✓ Piles_DialogueColoris.xml
✓ Vinci_TraitePeinture_fra.xml
✓ Vinci_TrattatoPittura_ITA.xml
✓ Zuccari_IdeaPittori.xml
✓ Zuccari_Lettera.xml
✓ Zuccari_OrigineProgressoAcademiaDissegno.xml

Terminé.
